# Legacy objective ablation — two variables, nothing else

ResNet-50 / CIFAR-10, BaCP + magnitude. Every knob is the **current** protocol; exactly two are reverted to the original design:

| knob | value | vs current |
|---|---|---|
| `contrastive_mode` | `legacy` — 2B x 2B SupCon+NTXent over `cat([student, teacher])` | **CHANGED** |
| `proj_mode` | `current` — per-model heads, student's trainable | **CHANGED** |
| tau | 0.15 | same |
| regime | 60 epochs, delta_T=88, cubic ramp to 80%, final 20% recovery | same |
| classifier head | pruned | same |
| dense checkpoint, seed, data, eval | identical | same |

So the only thing this measures is **the objective + head design**: the SimCLR-style composite with adaptive heads, against the rectangular CAP formulation with one frozen shared head. Records carry a `.legacy` key suffix and never collide with the main table.

The control arm (identical config, no overrides) is the main table's own `...s{sparsity}.magnitude.seed1` record, produced by the resnet50 notebook or by `91_compare_0p95`.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Run — 0.95 first, then 0.97 and 0.99

~25 min per sparsity. Each skips if its record already exists.

In [ ]:
GPU = 0
SEED = 1

LEGACY = dict(contrastive_mode='legacy',   # 2Bx2B SupCon+NTXent over cat([s,t])
              proj_mode='current')          # per-model heads, student's trainable

for s in (0.95, 0.97, 0.99):
    cell = nb.make_cell('resnet50', 'bacp', seed=SEED, pruner='magnitude',
                        sparsity=s, variant='legacy', **LEGACY)
    if s == 0.95:
        cfg = cell['config']
        print('config check — changed:',
              {k: cfg.get(k) for k in ('contrastive_mode', 'proj_mode')})
        print('config check — held:  ',
              {k: cfg.get(k) for k in ('tau', 'epochs', 'recovery_epochs',
                                       'delta_T', 'prune_task_head')})
    nb.run(cell, gpu=GPU)

## Verdict

Each arm against **its own** I.P. baseline — same scope, same regime, so the gains are directly comparable.

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']

def find(key):
    for f in glob.glob(os.path.join(root, 'runs', '*.json')):
        rec = json.load(open(f, encoding='utf-8'))
        if rec.get('experiment_group') == key:
            return rec
    return None

base = 'static.{p}.resnet50.cifar10.s{s}.magnitude.seed1'
print(f'{"sparsity":<9} {"I.P.":<8} {"current":<9} {"legacy":<9} '
      f'{"d(cur)":<8} {"d(leg)":<8} {"leg-cur"}')
for s in (0.95, 0.97, 0.99):
    ip  = find(base.format(p='prune', s=s))
    cur = find(base.format(p='bacp', s=s))
    leg = find(base.format(p='bacp', s=s) + '.legacy')
    def acc(r):
        return None if r is None else r.get('test_acc_pct')
    a_ip, a_cur, a_leg = acc(ip), acc(cur), acc(leg)
    f2 = lambda v: '-' if v is None else f'{v:.2f}'
    d_cur = '-' if (a_cur is None or a_ip is None) else f'{a_cur - a_ip:+.2f}'
    d_leg = '-' if (a_leg is None or a_ip is None) else f'{a_leg - a_ip:+.2f}'
    diff  = '-' if (a_leg is None or a_cur is None) else f'{a_leg - a_cur:+.2f}'
    print(f'{s:<9} {f2(a_ip):<8} {f2(a_cur):<9} {f2(a_leg):<9} '
          f'{d_cur:<8} {d_leg:<8} {diff}')
print()
print('d(cur), d(leg) = each objective\'s gain over the shared I.P. baseline.')
print('leg-cur > 0 -> the SimCLR composite + adaptive heads genuinely helps.')
print('leg-cur ~ 0 -> the objectives are equivalent; old-vs-new gaps came from')
print('              the other four changes (tau, regime, scope, eval).')